# Unit 9. Modelling Trends in Time

精算統計模型 (SOA Exam SRM/PA). The text of [unit09.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit09.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("lmtest")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/EuroExchange.csv", "data/frees/csv/HKExchange.csv", "data/frees/csv/InflationBond.csv", "data/frees/csv/LaborForcePR.csv", "data/frees/csv/PrescriptionDrug.csv", "data/frees/csv/SP500Daily.csv", "data/frees/csv/SP500Quarterly.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

## 1 Time Series and Trends

> **Reading.** FREES §7.1–7.2 (SRM reading list; not on the PA list). SRM learning outcomes 3(a), 3(c).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

### 1.1 Time series

**Definition 9.1** (Time series; FREES §7.1). A **stochastic process** is an ordered collection of random variables; one indexed by time, $y_1,\ldots,y_T$, observed once, is a **time series**. Data not ordered are **cross-sectional**. A regression $y_t=\beta_0+\beta_1x_t+\varepsilon_t$ of one series on another is a **causal model**.

**Theorem 9.2** (Spurious correlation; FREES §7.1). Let $x_t=a+\mu_xt+U_t$ and $y_t=b+\mu_yt+V_t$, $t=1,\ldots,T$, where $U_t=\sum_{j\leqslant t}u_j$ and $V_t=\sum_{j\leqslant t}v_j$ are partial sums of sequences of independent mean-zero variables with variances $\sigma_u^2$ and $\sigma_v^2$, the two sequences arbitrarily related. If $\mu_x\mu_y\neq0$, the sample correlation $r_T$ of $(x_t,y_t)_{t\leqslant T}$ satisfies $r_T\xrightarrow{\,p\,}\operatorname{sign}(\mu_x\mu_y)$.

**Proof.** Write $d_t=t-\bar t$, so that $D=\sum_td_t^2=T(T^2-1)/12$ and $D/T^3\to1/12$, and $x_t-\bar x=\mu_xd_t+(U_t-\bar U)$, $y_t-\bar y=\mu_yd_t+(V_t-\bar V)$.

**Step 1: the walks are small on the scale $T^{3/2}$.** By independence $\operatorname{\mathsf{E}} U_t^2=t\sigma_u^2$, so $\operatorname{\mathsf{E}} T^{-3}\sum_t(U_t-\bar U)^2\leqslant T^{-3}\sum_t\operatorname{\mathsf{E}} U_t^2=\sigma_u^2T(T+1)/(2T^3)\to0$, and Markov's inequality gives $A_T=T^{-3}\sum_t(U_t-\bar U)^2\xrightarrow{\,p\,}0$; likewise $B_T=T^{-3}\sum_t(V_t-\bar V)^2\xrightarrow{\,p\,}0$.

**Step 2: the three sums of products.** With $S_{xy}=\sum_t(x_t-\bar x)(y_t-\bar y)$,

$$
\frac{S_{xy}}{T^3}=\mu_x\mu_y\frac{D}{T^3}+\mu_x\frac{\sum_td_t(V_t-\bar V)}{T^3}
+\mu_y\frac{\sum_td_t(U_t-\bar U)}{T^3}+\frac{\sum_t(U_t-\bar U)(V_t-\bar V)}{T^3} ,
$$

and by Cauchy–Schwarz the second term is at most $|\mu_x|\sqrt{D/T^3}\sqrt{B_T}$, the third at most $|\mu_y|\sqrt{D/T^3}\sqrt{A_T}$ and the fourth at most $\sqrt{A_TB_T}$, all tending to $0$ in probability. So $S_{xy}/T^3\xrightarrow{\,p\,}\mu_x\mu_y/12$, and in the same way $S_{xx}/T^3\xrightarrow{\,p\,}\mu_x^2/12$ and $S_{yy}/T^3\xrightarrow{\,p\,}\mu_y^2/12$.

**Step 3.** $r_T=S_{xy}/\sqrt{S_{xx}S_{yy}}$ is a continuous function of the three ratios where the last two are positive, so the continuous mapping theorem of the prerequisites gives $r_T\xrightarrow{\,p\,}(\mu_x\mu_y/12)/(|\mu_x\mu_y|/12)$. ∎

**Example 9.1** (Spurious regression; FREES §7.1, Figure 7.1). FREES's construction, $x_t=0.5+x_{t-1}+\varepsilon_{x,t}$ and $y_t=0.5+y_{t-1}+\varepsilon_{y,t}$ from independent standard normal sequences with $x_0=y_0=0$, repeated $2000$ times for each length, with and without the drift $0.5$:

In [ ]:
set.seed(71)
spur <- function(T, mu)
  replicate(2000, cor(cumsum(mu + rnorm(T)), cumsum(mu + rnorm(T))))
sm <- sapply(c(50, 200, 1000), function(T) c(T = T,
  median_r_drift = median(spur(T, 0.5)),
  share_above_0.9_drift = mean(spur(T, 0.5) > 0.9),
  median_abs_r_no_drift = median(abs(spur(T, 0))),
  share_abs_above_0.5_no_drift = mean(abs(spur(T, 0)) > 0.5)))
round(t(sm), 3)

With drift the correlation of two independent series concentrates near $1$, as Theorem 9.2 says; FREES's realisation, $r=0.92$ at $T=50$, is typical. Without drift the theorem does not apply, and the correlation does not concentrate near $0$ either: the median absolute correlation does not shrink as $T$ grows.

### 1.2 Trends in time

**Definition 9.3** (Decomposition; FREES (7.1)–(7.2)). A series is decomposed into a **trend** $\mathit{Tr}_t$, a **seasonal** component $S_t$ and an irregular component $\varepsilon_t$, additively, $y_t=\mathit{Tr}_t+S_t+\varepsilon_t$, or multiplicatively, $y_t=\mathit{Tr}_t\times S_t+\varepsilon_t$; a purely multiplicative $y_t=\mathit{Tr}_t\times S_t\times\varepsilon_t$ with positive factors is additive in $\ln y_t$. The **linear trend in time model** is $y_t=\beta_0+\beta_1t+\varepsilon_t$ (FREES (7.3)), and polynomial trends add powers of $t$.

**Proposition 9.4** (Leverage in a linear trend; FREES §7.2). In the linear trend in time model fitted to $t=1,\ldots,T$, the leverage of observation $t$ is $h_t=1/T+(t-\bar t)^2/D$ with $D=T(T^2-1)/12$; it is largest at $t=1$ and $t=T$, where $h_t=1/T+3(T-1)/\{T(T+1)\}$, and smallest at the centre.

**Proof.** Unit 2's leverage of a simple regression with explanatory variable $x_t=t$ is $1/T+(t-\bar t)^2/\sum_s(s-\bar t)^2$, and $\sum_s(s-\bar t)^2=\sum_ss^2-T\bar t^2 =T(T+1)(2T+1)/6-T(T+1)^2/4=T(T^2-1)/12$. The leverage increases in $|t-\bar t|$, which is largest, $(T-1)/2$, at the ends, where $(T-1)^2/4$ divided by $D$ is $3(T-1)/\{T(T+1)\}$. ∎

**Remark** (FREES §7.2). So a trend fitted by least squares gives the most weight to the first and last observations (at $T=100$, each has leverage $0.0394$, about twice the average $2/T$); the first ones matter for the fit as much as the most recent, which are the ones a forecaster cares about.

**Example 9.2** (Hong Kong exchange rates; FREES §7.2, Figure 7.3). FREES's $T=502$ daily rates, Hong Kong dollars per US dollar, with a quadratic trend:

In [ ]:
hk <- read.csv("data/frees/csv/HKExchange.csv"); hk$t <- seq_len(nrow(hk))
mhk <- lm(EXHKUS ~ t + I(t^2), data = hk)
signif(coef(summary(mhk))[, c(1, 3)], 6)
c(R2 = summary(mhk)$r.squared, s_y = sd(hk$EXHKUS), s = summary(mhk)$sigma,
  R2_from_s = 1 - (summary(mhk)$sigma / sd(hk$EXHKUS))^2 * 499 / 501,
  forecast_503 = unname(predict(mhk, data.frame(t = 503))))
hk$DATE[c(1, 502)]

The estimates, the $t$-statistics ($8531.9$, $-44.0$, $51.2$), $s_y=0.0183$, $s=0.0068$ and the forecast $7.8208$ at $t=503$ are FREES's (FREES truncates $b_1=-3.6891\times10^{-4}$ to $-3.68\times10^{-4}$). Its $R^2=92.9\%$ is not: from its own $s$ and $s_y$, $R^2=1-(s/s_y)^2(T-3)/(T-1)=0.862$, which `lm` confirms. The file runs from 1 April 2005 to 30 March 2007, not to 31 May 2007 as FREES states, which is why FREES dates $t=503$ in April 2007.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.4, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
plot(hk$t, hk$EXHKUS, type = "l", xlab = "t (trading day)", ylab = "HK$ per US$",
     main = "series and quadratic trend")
lines(hk$t, fitted(mhk), col = "#B45309", lwd = 2)
plot(hk$t, residuals(mhk), type = "l", xlab = "t (trading day)", ylab = "residual",
     main = "residuals"); abline(h = 0, col = "#B45309")
par(op)

*Figure 9.1.* Hong Kong exchange rates with the fitted quadratic trend (FREES Figure 7.3), and the residuals, which stay above or below zero for long stretches.

**Remark** (Change points; FREES (7.4)). An indicator $z_t=\mathbb{1}\{t\geqslant t_0\}$ for a known change point $t_0$ in $y_t=\beta_0+\beta_1z_t+\varepsilon_t$ makes $\beta_1$ the shift in the mean at $t_0$. A **regime-switching** model (Hardy's, in FREES §7.2) lets the regime follow an unobserved Markov chain, with the transition probabilities estimated.

### 1.3 Seasonal trends

**Proposition 9.5** (Seasonal indicators; FREES §7.2). Let each $t$ belong to one of $m$ seasons, every season occurring, and fit $y_t=\sum_{j=1}^m\alpha_j\mathbb{1}\{t\in\text{season }j\}+\varepsilon_t$ by least squares. Then $\hat\alpha_j$ is the mean of the observations in season $j$. With a trend $\beta_1t$ added, the $\hat\alpha_j$ are the season means of $y_t-\hat\beta_1t$.

**Proof.** The indicator columns are orthogonal, with $\mathbf{X}'\mathbf{X}=\operatorname{diag}(n_1,\ldots,n_m)$ and $j$th entry of $\mathbf{X}'\mathbf{Y}$ the sum of the season-$j$ observations; so $\hat\alpha_j$ is their mean. With the trend, for the fitted $\hat\beta_1$ the normal equations for the $\alpha_j$ are those of the indicator regression of $y_t-\hat\beta_1t$. ∎

**Example 9.3** (Prescription drug costs; FREES §7.2, with the data of FREES §9.3). FREES's monthly cost per claim of the New Jersey prescription drug plan, August 1986 to March 1992 ($T=68$), with a linear trend and month indicators (FREES fits trigonometric seasonals to this series in its chapter 9):

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
rx <- read.csv("data/frees/csv/PrescriptionDrug.csv")
rx$month <- factor(rx$MONTH)
m_trend <- lm(COST_CLM ~ TIME, data = rx)
m_season <- lm(COST_CLM ~ TIME + month - 1, data = rx)
anova(m_trend, m_season)
detr <- rx$COST_CLM - coef(m_season)[["TIME"]] * rx$TIME
round(max(abs(tapply(detr, rx$month, mean) - coef(m_season)[-1])), 12)

The month indicators reduce the residual sum of squares from $35.6$ to $21.2$, $F=3.4$ on $11$ and $55$ degrees of freedom ($p=0.0012$ under A1–A5), and their coefficients are the month means of the detrended series, as Proposition 9.5 states.

**Remark** (Seasonal adjustment; FREES §7.2). Removing the seasonal pattern from a series is **seasonal adjustment**. It suits public policy, where interest is in the adjusted series itself: government agencies report seasonally adjusted manufacturing revenues so that growth is not confused with holiday and weather patterns. For most actuarial and risk management applications the interest is in forecasting the whole series, seasonal part included, so the seasonal terms are modelled, as above, not removed.

**Remark** (Reliability of forecasts; FREES §7.2). Time series forecasts are sometimes called **naive**: they extrapolate one series and ignore other information the forecaster may have, which makes them an objective benchmark for expert forecasts. A forecast should come with a sense of its reliability. One way is a set of low–intermediate–high projections under different assumptions, which reflects the spread of expert opinion; another is a prediction interval, which reflects the innovation and estimation uncertainty of a model assumed correct and has a probability interpretation; for a random walk or a linear trend in time it widens as the forecast horizon grows. A regression on functions of time fits the whole series and gives the most weight to the observations at its two ends, the beginning as much as the end (Proposition 9.4), while a forecaster cares most about the recent past; this motivates the smoothing methods of Unit 10.

## 2 Stationarity and Random Walks

> **Reading.** FREES §7.3. SRM learning outcome 3(a).

### 2.1 Stationarity and white noise

**Definition 9.6** (Stationarity, white noise; FREES §7.3). A series with finite variances is **weakly stationary** if $\operatorname{\mathsf{E}} y_t$ does not depend on $t$ and $\operatorname{cov}(y_s,y_t)$ depends only on $|t-s|$; it is **strictly stationary** if the joint law of $(y_{t+1},\ldots,y_{t+m})$ does not depend on $t$, for every $m$. FREES's **white noise** is an independent and identically distributed series.

**Proposition 9.7** (Properties; FREES §7.3). A weakly stationary series has constant variance $\gamma_0=\operatorname{cov}(y_t,y_t)$. A strictly stationary series with finite variances is weakly stationary; white noise with finite variance is both.

**Proof.** Take $s=t$ in the definition. Under strict stationarity $y_t$ has the law of $y_1$ and $(y_s,y_t)$ that of $(y_1,y_{1+|t-s|})$, so means and covariances are those of the shifted pair. An independent identically distributed series has every joint law equal to the product of copies of one law, which does not depend on $t$. ∎

**Theorem 9.8** (Forecasting white noise; FREES (7.5)). Let $y_1,\ldots,y_T,y_{T+l}$ be independent $N(\mu,\sigma^2)$, $T\geqslant2$. Then

$$
\frac{y_{T+l}-\bar y}{s_y\sqrt{1+1/T}}\sim t_{T-1} ,
$$

so $\bar y\pm t_{T-1,1-\alpha/2}\,s_y\sqrt{1+1/T}$ covers $y_{T+l}$ with probability $1-\alpha$, for every lead time $l$.

**Proof.** Let $\mathbf{Y}=(y_1,\ldots,y_T)'$ and $\mathbf{M}_0=\mathbf{I}-T^{-1}\mathbf1\mathbf1'$. The vector $(y_{T+l},\bar y,(\mathbf{M}_0\mathbf{Y})')'$ is jointly normal and $\operatorname{cov}(\bar y,\mathbf{M}_0\mathbf{Y})=\sigma^2T^{-1}\mathbf1'\mathbf{M}_0=\mathbf{0}$, while $y_{T+l}$ is independent of $\mathbf{Y}$; so $y_{T+l}$, $\bar y$ and $\mathbf{M}_0\mathbf{Y}$ are independent by the prerequisites' theorem **uncorrelated blocks are independent**. Hence $y_{T+l}-\bar y\sim N(0,\sigma^2(1+1/T))$ is independent of $(T-1)s_y^2/\sigma^2=(\mathbf{Y}-\mu\mathbf1)'\mathbf{M}_0(\mathbf{Y}-\mu\mathbf1)/\sigma^2$, which is $\chi^2_{T-1}$ by the prerequisites' theorem **chi-square quadratic forms**, $\mathbf{M}_0$ being symmetric idempotent of trace $T-1$. The statistic of the theorem is $(y_{T+l}-\bar y)/\{\sigma\sqrt{1+1/T}\}$, standard normal by the prerequisites' corollary **standardising a normal variable**, divided by $\sqrt{\{(T-1)s_y^2/\sigma^2\}/(T-1)}$, so it is $t_{T-1}$ by the prerequisites' definition of the $t$ law. ∎

**Remark** (FREES §7.3). For large $T$ FREES uses $\bar y\pm2s_y$, ignoring the parameter uncertainty in $\bar y$ and $s_y$ and keeping the **innovation uncertainty**. Reducing a series to white noise is **filtering**.

### 2.2 The random walk

**Definition 9.9** (Random walk; FREES (7.6)–(7.7)). Given white noise $c_1,c_2,\ldots$ with mean $\mu_c$ and variance $\sigma_c^2$ and an initial level $y_0$, the **random walk** is $y_t=y_{t-1}+c_t$, that is $y_t=y_0+c_1+\cdots+c_t$, the partial sums; $\mu_c$ is the **drift**.

**Proposition 9.10** (Moments; FREES §7.4, exercise 7.1). $\operatorname{\mathsf{E}} y_t=y_0+t\mu_c$, $\operatorname{var} y_t=t\sigma_c^2$ and $\operatorname{cov}(y_s,y_t)=\min(s,t)\,\sigma_c^2$. The random walk is not stationary when $\sigma_c^2>0$, its variance growing with $t$ (**nonstationary in the variance**: $\operatorname{var} y_t$ depends on $t$); it is **nonstationary in the mean** ($\operatorname{\mathsf{E}} y_t$ depends on $t$) when $\mu_c\neq0$.

**Proof.** Linearity of expectation gives the mean; the $c_j$ are independent, so $\operatorname{var} y_t=\sum_{j\leqslant t}\operatorname{var} c_j$ and, for $s\leqslant t$, $\operatorname{cov}(y_s,y_t)=\operatorname{var}(c_1+\cdots+c_s)=s\sigma_c^2$. A constant variance would need $t\sigma_c^2=\sigma_c^2$ for every $t$. ∎

**Example 9.4** (A game of dice; FREES §7.3–7.4, Table 7.1). Paying \$7 per roll of two dice and receiving their sum $c^*_t$ gives winnings $c_t=c^*_t-7$ with mean $0$ and variance $35/6$, each die having variance $(1^2+\cdots+6^2)/6-3.5^2=35/12$; from $y_0=\$100$ the capital is a random walk. FREES's first five rolls give $y_1,\ldots,y_5=103,105,105,103,103$.

## 3 Inference Using Random Walk Models

> **Reading.** FREES §7.4. SRM learning outcomes 3(a), 3(c).

### 3.1 Forecasting

**Theorem 9.11** (Forecasting a random walk; FREES §7.4, exercise 7.2). Let $c_1,\ldots,c_m$ be the observed differences of a random walk up to time $T$, and $c_{T+1},c_{T+2},\ldots$ the future ones, all independent with mean $\mu_c$ and variance $\sigma_c^2$, and let $\hat y_{T+l}=y_T+l\bar c$. Then

(i) $y_{T+l}-\hat y_{T+l}=\sum_{j=1}^l(c_{T+j}-\bar c)$, with mean $0$ and variance $l\sigma_c^2(1+l/m)$, which divided by $l\sigma_c^2$ tends to $1$ as $m\to\infty$;

(ii) if the $c$'s are normal and $m\geqslant2$, $(y_{T+l}-\hat y_{T+l})/\{s_c\sqrt{l(1+l/m)}\}\sim t_{m-1}$, $s_c$ the sample standard deviation of $c_1,\ldots,c_m$.

**Proof.** (i) $y_{T+l}=y_T+\sum_{j\leqslant l}c_{T+j}$, so the error is $\sum_jc_{T+j}-l\bar c=\sum_j(c_{T+j}-\bar c)$. Its mean is $l\mu_c-l\mu_c$. The future $c$'s are independent of $\bar c$, so its variance is $l\sigma_c^2+l^2\operatorname{var}\bar c =l\sigma_c^2+l^2\sigma_c^2/m$.

(ii) As in the proof of Theorem 9.8, $\sum_jc_{T+j}$, $\bar c$ and the deviations $c_i-\bar c$ are independent, so the error is $N(0,l\sigma_c^2(1+l/m))$ and independent of $(m-1)s_c^2/\sigma_c^2\sim\chi^2_{m-1}$; divide. ∎

**Example 9.5** (Dice, continued; FREES §7.4). FREES reports $y_{50}=93$, so $\bar c=-7/50=-0.14$, and $s_c=2.703$; its forecast of $y_{60}$ and approximate interval, and the exact interval of Theorem 9.11:

In [ ]:
l <- 10; m <- 50; cbar <- -0.14; sc <- 2.703
round(c(sd_c_true = sqrt(35 / 6), forecast = 93 + l * cbar,
        half_width_frees = 2 * sc * sqrt(l),
        half_width_exact = qt(0.975, m - 1) * sc * sqrt(l * (1 + l / m))), 3)

FREES's forecast $91.6$ and interval $91.6\pm17.1=(74.5,108.7)$ are reproduced; the exact $95\%$ interval is wider, by the factor $\sqrt{1+l/m}=1.095$ and the $t_{49}$ quantile. FREES computes $s_c$ “using the changes $c_2,\ldots,c_T$”, although $y_0=100$ is known and gives $c_1$ as well. (The exact interval treats FREES's $s_c$ as the standard deviation of all $50$ changes; computed from $c_2,\ldots,c_{50}$ it calls for the $t_{48}$ quantile, which changes the half width in the second decimal.)

**Remark** (FREES §7.4). FREES's approximate interval $y_T+l\bar c\pm2s_c\sqrt l$ keeps $l\sigma_c^2$ and drops the term $l^2\sigma_c^2/m$ from the uncertainty in $\bar c$. FREES's width grows like $\sqrt l$; the exact one, proportional to $\sqrt{l+l^2/m}$, grows like $\sqrt l$ while $l$ is small relative to $m$ and like $l/\sqrt m$ beyond. At $l=m$ the dropped term equals the kept one.

**Example 9.6** (Labor force participation rates; FREES §7.4, Figure 7.7). FREES's rates for married women aged $20$–$44$ with a child under six, 1968–98 ($T=31$), and the random-walk forecasts for 1999–2002:

In [ ]:
lf <- read.csv("data/frees/csv/LaborForcePR.csv"); msc <- lf$MSC6U; dl <- diff(msc)
c(last = tail(msc, 1), cbar = mean(dl), s_c = sd(dl), sd_series = sd(msc))
L <- 1:4; mm <- length(dl)
round(cbind(year = 1998 + L, forecast = tail(msc, 1) + mean(dl) * L,
  half_frees = 2 * sd(dl) * sqrt(L),
  half_exact = qt(0.975, mm - 1) * sd(dl) * sqrt(L * (1 + L / mm))), 4)

FREES's $\mathit{LFPR}_{31}=0.6407$, $\bar c=0.0121$ and $s_c=0.0101$, hence its interval $0.6407+0.0121l\pm0.0201\sqrt l$, are reproduced (FREES's $0.0202$ doubles the rounded $s_c$). The standard deviation of the series is $0.1237$, not FREES's $0.1197$.

### 3.2 Identifying stationarity and random walks

**Definition 9.12** (Control charts; FREES §7.4). A **control chart** is a time series plot with the **control limits** $\bar y\pm3s_y$; **retrospective** when $\bar y$ and $s_y$ use all the data, prospective when they use an early, in-control stretch. An $\bar X$ chart plots the means of successive groups of a few observations, an $R$ chart their ranges, each with its control limits.

**Proposition 9.13** (The variability of a random walk's level; FREES §7.4). Let $y_t=y_0+\mu_ct+u_t$, $t=1,\ldots,T$, with $u_t=\varepsilon_1+\cdots+\varepsilon_t$ the partial sums of independent mean-zero $\varepsilon_j$ of variance $\sigma^2$. Then

$$
\operatorname{\mathsf{E}}\sum_{t=1}^T(y_t-\bar y)^2=\mu_c^2\frac{T(T^2-1)}{12}+\sigma^2\frac{T^2-1}{6} ,
$$

so $\operatorname{\mathsf{E}} s_y^2=\mu_c^2T(T+1)/12+\sigma^2(T+1)/6$, while the differences $c_t=\mu_c+\varepsilon_t$ have $\operatorname{\mathsf{E}} s_c^2=\sigma^2$.

**Proof.** With $d_t=t-\bar t$, $y_t-\bar y=\mu_cd_t+(u_t-\bar u)$ and $\operatorname{\mathsf{E}}\sum_td_t(u_t-\bar u)=0$, so the expectation is $\mu_c^2\sum_td_t^2$, computed in Proposition 9.4, plus $\operatorname{\mathsf{E}}\sum_t(u_t-\bar u)^2=\sum_t\operatorname{\mathsf{E}} u_t^2 -T\operatorname{var}\bar u$. Here $\sum_t\operatorname{\mathsf{E}} u_t^2=\sigma^2T(T+1)/2$, and by the proof of Proposition 9.10, which uses only independence and the common variance, $\operatorname{var}\bar u=T^{-2}\sigma^2\sum_{s,t\leqslant T}\min(s,t)$. Counting, for each $k$, the $(T-k+1)^2$ pairs with $\min(s,t)\geqslant k$, $\sum_{s,t}\min(s,t)=\sum_{k=1}^T(T-k+1)^2=\sum_{j=1}^Tj^2=T(T+1)(2T+1)/6$. So $\operatorname{\mathsf{E}}\sum_t(u_t-\bar u)^2=\sigma^2(T+1)\{T/2-(2T+1)/6\}=\sigma^2(T^2-1)/6$. Divide by $T-1$. The differences are independent with variance $\sigma^2$, and a sample variance is unbiased. ∎

**Remark** (Identifying a random walk; FREES §7.4). FREES's three indications: a control chart showing a linear trend and growing variability (Proposition 9.10); differences that look like white noise (Definition 9.9); and a standard deviation of the series far above that of the differences, by Proposition 9.13 a factor of order $\sqrt T$ even without drift. On the labor force rates (Figure 9.2), $0.1237$ against $0.0101$ (Example 9.6).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.4, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
cc <- function(x, yr, main) { plot(yr, x, type = "o", pch = 16, cex = 0.5, xlab = "year",
  ylab = "", main = main, ylim = range(x, mean(x) + c(-3, 3) * sd(x)))
  abline(h = mean(x) + c(-3, 0, 3) * sd(x), lty = c(2, 1, 2), col = "#B45309") }
cc(msc, lf$YEAR, "LFPR"); cc(dl, lf$YEAR[-1], "differences")
par(op)

*Figure 9.2.* Retrospective control charts, with limits $\bar y\pm3s_y$ (dashed), of the labor force participation rates (FREES Figure 7.7) and of their differences.

### 3.3 Random walk against linear trend

**Proposition 9.14** (Two models with a linear mean; FREES (7.8)–(7.9)). A random walk with drift is $y_t=y_0+\mu_ct+u_t$ with $u_t=\sum_{j\leqslant t}\varepsilon_j$, $\varepsilon_j=c_j-\mu_c$ white noise; the linear trend model is $y_t=\beta_0+\beta_1t+\eta_t$. Both have a linear mean. The trend model's error $\eta_t$ is white noise, stationary; the random walk's error $u_t$ is itself a random walk, with variance $t\sigma_c^2$. If $\mu_c=0$ the random walk's mean is constant, so it competes only with the trend model's special case $\beta_1=0$.

**Proof.** Substitute $c_j=\mu_c+\varepsilon_j$ in $y_t=y_0+\sum_{j\leqslant t}c_j$; the variance of $u_t$ is Proposition 9.10 with drift $0$. ∎

## 4 Filtering to Achieve Stationarity

> **Reading.** FREES §7.5. SRM learning outcome 3(a).

**Proposition 9.15** (Differences of logarithms; FREES §7.5). For $|x|<1$, $|\ln(1+x)-x|\leqslant x^2/\{2(1-|x|)^2\}$. So, for a positive series with $\mathit{pchange}_t=y_t/y_{t-1}-1$ and $|\mathit{pchange}_t|<1$, $\ln y_t-\ln y_{t-1}=\ln(1+\mathit{pchange}_t)$ differs from the proportional change by at most $\mathit{pchange}_t^2/\{2(1-|\mathit{pchange}_t|)^2\}$.

**Proof.** Taylor's theorem of the prerequisites with Lagrange remainder gives $\ln(1+x)=x-x^2/\{2(1+\xi)^2\}$ for some $\xi$ between $0$ and $x$, and $1+\xi\geqslant1-|x|>0$. ∎

**Example 9.7** (The S&P 500 quarterly index; FREES §7.5, Figure 7.9). FREES's quarterly index, 1936–2006: the four series of its Figure 7.9 (Figure 9.3) and the standard deviations of the first and last thirds of each, a crude check of constant variability:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
sp <- read.csv("data/frees/csv/SP500Quarterly.csv")
x <- list(index = sp$SPINDEX, difference = diff(sp$SPINDEX),
          log_index = log(sp$SPINDEX),
          difference_of_logs = diff(log(sp$SPINDEX)))
round(t(sapply(x, function(v) { k <- floor(length(v) / 3)
  c(sd_first_third = sd(head(v, k)), sd_last_third = sd(tail(v, k))) })), 4)
pc <- diff(sp$SPINDEX) / head(sp$SPINDEX, -1)
c(max_abs_log_change = max(abs(x$difference_of_logs)),
  max_bound = max(pc^2 / (2 * (1 - abs(pc))^2)),
  max_gap = max(abs(x$difference_of_logs - pc)))

The differences have a standard deviation $36$ times larger in the last third than in the first ($63.70$ against $1.78$); the differences of logarithms do not grow in this way ($0.0882$ and $0.0775$), and FREES takes them as the stationary series. The largest quarterly log change is $0.31$ in absolute value, and over all quarters the bound of Proposition 9.15 on its distance from the proportional change is at most $0.158$ (the actual distance is at most $0.053$): a log difference is a proportional change only for small changes.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.3)
op <- par(mfrow = c(2, 2), mar = c(2.6, 3.4, 1.8, 0.6), mgp = c(1.9, 0.6, 0), cex = 0.7)
plot(sp$YEAR, sp$SPINDEX, type = "l", xlab = "", ylab = "", main = "index")
plot(sp$YEAR[-1], x$difference, type = "l", xlab = "", ylab = "", main = "difference")
plot(sp$YEAR, x$log_index, type = "l", xlab = "year", ylab = "", main = "log index")
plot(sp$YEAR[-1], x$difference_of_logs, type = "l", xlab = "year", ylab = "",
     main = "difference of logs")
par(op)

*Figure 9.3.* The S&P 500 quarterly index, 1936–2006 (FREES Figure 7.9).

## 5 Forecast Evaluation

> **Reading.** FREES §7.6. SRM learning outcome 3(c).

**Definition 9.16** (Out-of-sample validation; FREES §7.6). Split $y_1,\ldots,y_T$ into a model development subsample $t\leqslant T_1$ and a validation subsample $T_1<t\leqslant T_1+T_2$; fit each candidate to the first, forecast each validation $y_t$ from the fitted model and the data up to $t-1$, and form the **one-step forecast residuals** $e_t=y_t-\hat y_t$. With sums over the validation subsample,

$$
\begin{gather*}
\text{ME}=\frac1{T_2}\sum e_t ,\qquad
\text{MSE}=\frac1{T_2}\sum e_t^2 ,\qquad
\text{MAE}=\frac1{T_2}\sum|e_t| ,\\
\text{MPE}=\frac{100}{T_2}\sum\frac{e_t}{y_t} ,\qquad
\text{MAPE}=\frac{100}{T_2}\sum\Bigl|\frac{e_t}{y_t}\Bigr| .
\end{gather*}
$$

The candidate whose statistics are smallest in absolute value is chosen.

**Example 9.8** (Labor force rates: trend against random walk; FREES §7.6, Table 7.2). Both models fitted to 1968–94 ($T_1=27$) and evaluated on 1995–98 ($T_2=4$); for the random walk, the one-step forecasts $y_{t-1}+\bar c$ of the definition and the forecasts from 1994 at leads $1$ to $4$ that FREES's script computes; the percentage statistics without the factor $100$, as FREES prints them:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
yr <- lf$YEAR; tt <- 1:31
lin <- lm(msc ~ tt, subset = yr %in% 1968:1994); cb <- mean(dl[yr[-1] %in% 1969:1994])
y58 <- msc[yr %in% 1995:1998]
fc <- list(linear_trend = coef(lin)[[1]] + coef(lin)[[2]] * tt[yr %in% 1995:1998],
           random_walk_one_step = msc[yr %in% 1994:1997] + cb,
           random_walk_script = msc[yr == 1994] + cb * (1:4))
round(c(coef(lin), cbar = cb), 4)
round(t(sapply(fc, function(f) { e <- y58 - f
  c(ME = mean(e), MPE = mean(e / y58), MSE = mean(e^2), MAE = mean(abs(e)),
    MAPE = mean(abs(e / y58))) })), 4)

The fitted $0.2574+0.0145t$ and $\bar c=0.0132$ are FREES's, and the linear trend row of Table 7.2 is reproduced. FREES's random walk row ($-0.0007$, $0.0012$, $0.0001$, $0.0115$, $0.0180$) is reproduced by neither forecast; both give smaller errors than the trend on every statistic, so FREES's choice of the random walk stands. FREES's $\text{MPE}$ and $\text{MAPE}$ are fractions, not the percentages of its definitions.

## 6 Autocorrelations

> **Reading.** FREES §8.1. SRM learning outcome 3(a).

**Definition 9.17** (Autocorrelation; FREES §8.1). For a weakly stationary series the **autocovariance** at lag $k$ is $\gamma_k=\operatorname{cov}(y_t,y_{t-k})$ and the **autocorrelation** is $\rho_k=\gamma_k/\gamma_0$. For data $y_1,\ldots,y_T$ with $a_t=y_t-\bar y$, the **lag-$k$ autocorrelation statistic** is

$$
r_k=\frac{\sum_{t=k+1}^Ta_{t-k}a_t}{\sum_{t=1}^Ta_t^2} , \qquad k=1,2,\ldots
$$

**Proposition 9.18** (Range, and the Durbin–Watson statistic; FREES §8.1, exercise 8.2). If the $y_t$ are not all equal, $|r_k|\leqslant1$, and the Durbin–Watson statistic $\text{DW}=\sum_{t=2}^T(y_t-y_{t-1})^2/\sum_{t=1}^Ta_t^2$ satisfies exactly

$$
\text{DW}=2(1-r_1)-\frac{a_1^2+a_T^2}{\sum_ta_t^2} .
$$

**Proof.** By Cauchy–Schwarz, $|\sum_{t>k}a_{t-k}a_t|\leqslant(\sum_{t>k}a_{t-k}^2)^{1/2} (\sum_{t>k}a_t^2)^{1/2}\leqslant\sum_ta_t^2$. For $\text{DW}$, $y_t-y_{t-1}=a_t-a_{t-1}$, and $\sum_{t\geqslant2}(a_t-a_{t-1})^2=\sum_{t\geqslant2}a_t^2+\sum_{t\geqslant2}a_{t-1}^2 -2\sum_{t\geqslant2}a_ta_{t-1}=(\sum_ta_t^2-a_1^2)+(\sum_ta_t^2-a_T^2)-2r_1\sum_ta_t^2$. ∎

**Proposition 9.19** (Autocorrelations of white noise vanish). If $y_1,y_2,\ldots$ are independent and identically distributed with $\operatorname{\mathsf{E}} y_t^4<\infty$ and variance $\sigma^2>0$, then $r_k\xrightarrow{\,p\,}0$ for each fixed $k\geqslant1$.

**Proof.** Write $z_t=y_t-\mu$, so $a_t=z_t-\bar z$. For $k\geqslant1$ the products $z_tz_{t-k}$, $t=k+1,\ldots,T$, have mean $0$ and are uncorrelated, since $\operatorname{\mathsf{E}}[z_tz_{t-k}z_sz_{s-k}]=0$ for $s<t$ ($z_t$ appears once and is independent of the rest); each has variance $\sigma^4$, so $T^{-1}\sum_tz_tz_{t-k}\xrightarrow{\,p\,}0$ by Chebyshev's inequality. Likewise $\bar z\xrightarrow{\,p\,}0$, and $T^{-1}\sum_tz_t^2\xrightarrow{\,p\,}\sigma^2$ because the $z_t^2$ are independent with finite variance. Expanding,

$$
\frac1T\sum_{t>k}a_{t-k}a_t=\frac1T\sum_{t>k}z_{t-k}z_t-\bar z\,\frac1T\sum_{t>k}(z_t+z_{t-k})
+\frac{T-k}{T}\bar z^2\xrightarrow{\,p\,}0 ,
$$

each average $T^{-1}\sum_{t>k}z_t$, $T^{-1}\sum_{t>k}z_{t-k}$ tending to $0$ in probability by Chebyshev's inequality, so the products tend to $0$ by the continuous mapping theorem, and $T^{-1}\sum_ta_t^2= T^{-1}\sum_tz_t^2-\bar z^2\xrightarrow{\,p\,}\sigma^2$. The ratio converges to $0/\sigma^2$ by the continuous mapping theorem. ∎

**Example 9.9** (Inflation bond index; FREES §8.1, Tables 8.1–8.3).

In [ ]:
ib <- read.csv("data/frees/csv/InflationBond.csv"); yb <- ib$INFBOND
round(c(T = length(yb), mean = mean(yb), median = median(yb), sd = sd(yb),
        min = min(yb), max = max(yb)), 3)
head(yb, 5)
rk <- function(y, k) { a <- y - mean(y); sum(a[-(1:k)] * head(a, -k)) / sum(a^2) }
rbind(manual = sapply(1:5, rk, y = yb),
      acf = acf(yb, lag.max = 5, plot = FALSE)$acf[2:6])

FREES's Tables 8.1 and 8.2 are reproduced, and its autocorrelations $0.814$, $0.632$, $0.561$, $0.447$ at lags $1$ to $4$; at lag $5$ the statistic is $0.367$, not FREES's $0.267$. `acf` computes Definition 9.17.

## 7 Computation in R

> **Reading.** FREES chapters 7–8 scripts.

**Example 9.10** (Coverage of the random-walk forecast intervals; Theorem 9.11). Normal random walks with $m$ observed differences, $10{,}000$ replications: the share of $y_{T+l}$ inside FREES's $y_T+l\bar c\pm2s_c\sqrt l$ and inside the exact $t$ interval:

In [ ]:
set.seed(97)
cov_rw <- function(m, l, R = 10000) {
  inside <- replicate(R, { c0 <- rnorm(m, 0.3, 1); fut <- sum(rnorm(l, 0.3, 1))
    e <- fut - l * mean(c0); s <- sd(c0)
    c(frees = abs(e) <= 2 * s * sqrt(l),
      exact = abs(e) <= qt(0.975, m - 1) * s * sqrt(l * (1 + l / m))) })
  rowMeans(inside) }
round(rbind(`m=10, l=1` = cov_rw(10, 1), `m=10, l=10` = cov_rw(10, 10),
            `m=50, l=10` = cov_rw(50, 10), `m=200, l=10` = cov_rw(200, 10)), 3)

The exact interval covers $95\%$ at every $m$ and $l$; FREES's falls short when $m$ is small or $l$ is not small relative to $m$.

**Example 9.11** (Durbin–Watson against `lmtest`; Proposition 9.18). For the residuals $a_t$ of the Hong Kong trend (their mean is $0$):

In [ ]:
a <- residuals(mhk)
dw <- sum(diff(a)^2) / sum(a^2); r1 <- rk(a, 1)
signif(c(DW = dw, identity = 2 * (1 - r1) - unname(a[1]^2 + a[502]^2) / sum(a^2),
         two_times_1_minus_r1 = 2 * (1 - r1),
         lmtest = unname(lmtest::dwtest(mhk)$statistic)), 6)

## 8 Exercises

**Exercise 9.1** (FREES exercise 7.1). For a random walk, the partial sum of white noise with mean $\mu_c$ and variance $\sigma_c^2$, show that $\operatorname{\mathsf{E}} y_t=y_0+t\mu_c$ and $\operatorname{var} y_t=t\sigma_c^2$.

**Solution.** Proposition 9.10: $y_t=y_0+c_1+\cdots+c_t$ with independent $c_j$, so the mean is $y_0+t\mu_c$ and the variance the sum of the $t$ variances. ■

**Exercise 9.2** (FREES exercise 7.2). For a random walk, show that (a) the $l$-step forecast error is $\sum_{j=1}^l(c_{T+j}-\bar c)$, and (b) its approximate variance is $l\sigma_c^2$.

**Solution.** Theorem 9.11(i): the error is exactly $\sum_j(c_{T+j}-\bar c)$, with variance $l\sigma_c^2(1+l/m)$ for $m$ observed differences, and the variance divided by $l\sigma_c^2$ tends to $1$ as $m\to\infty$, so $l\sigma_c^2$, the variance of the future changes alone, is its approximation. ■

**Exercise 9.3** (FREES exercise 7.3). For FREES's $T=699$ daily euro rates (euros per US dollar, April 2005 to January 2008): (a)(i) define a stationary series; (ii) is the series stationary? (b) Fit a quadratic trend: (i) cite basic statistics of the fit; (ii) describe the residual patterns; (iii) forecast at $\mathit{TIME}=702$. (c) Model the differences as white noise: (i) name the model for the series; (ii) with $\mathit{EURO}_{699}=0.6795$, forecast at $\mathit{TIME}=702$; (iii) give a $95\%$ prediction interval.

**Solution.**

In [ ]:
eu <- read.csv("data/frees/csv/EuroExchange.csv", quote = ""); E <- eu$exeuus
TIME <- seq_along(E); qe <- lm(E ~ TIME + I(TIME^2)); dE <- diff(E)
signif(coef(summary(qe))[, c(1, 3)], 5)
round(c(T = length(E), first = E[1], R2 = summary(qe)$r.squared,
        s = summary(qe)$sigma, s_y = sd(E),
        quadratic_702 = unname(predict(qe, data.frame(TIME = 702))),
        last = tail(E, 1), cbar = mean(dE), s_c = sd(dE),
        rw_702 = tail(E, 1) + 3 * mean(dE), half_frees = 2 * sd(dE) * sqrt(3),
        half_exact = qt(0.975, 697) * sd(dE) * sqrt(3 * (1 + 3 / 698))), 5)
runs <- rle(sign(residuals(qe)))$lengths
round(c(sign_runs = length(runs), longest_run = max(runs),
        r1_resid = rk(residuals(qe), 1)), 3)

(a)(i) Definition 9.6. (ii) No: the level drifts from $0.7754$ to $0.6795$ and stays away from any fixed level for long stretches, the behaviour of the random walk of part (c), whose variance grows with $t$ (Proposition 9.10). (b)(i) $R^2=0.873$, $s=0.0164$ against $s_y=0.0459$, and every $|t|>10$. (ii) As for the Hong Kong rates (Figure 9.1), the series stays on one side of the fitted curve for long stretches: the residuals form only $44$ runs of one sign in $699$ days, the longest $116$ days, and their lag-1 autocorrelation is $0.973$. (iii) $0.6703$. (c)(i) A random walk. (ii) $0.6795+3\bar c=0.6791$. (iii) FREES's approximation gives $0.6791\pm0.0125$, that is $(0.6665,0.6916)$; the exact interval of Theorem 9.11 is $0.6791\pm0.0123$, slightly narrower, because $t_{697,0.975}=1.96<2$ outweighs the factor $\sqrt{1+3/698}$. ■

**Exercise 9.4** (FREES exercise 8.1). Yields $0.09$, $0.08$, $0.09$, $0.12$, $-0.03$ in five consecutive years: find $r_1$ and $r_2$.

**Solution.** $\bar y=0.07$ and $a_t=0.02,0.01,0.02,0.05,-0.10$, $\sum a_t^2=0.0134$. Then $r_1=(0.0002+0.0002+0.0010-0.0050)/0.0134=-0.0036/0.0134=-0.2687$ and $r_2=(0.0004+0.0005-0.0020)/0.0134=-0.0821$: ■

In [ ]:
yy <- c(0.09, 0.08, 0.09, 0.12, -0.03); round(c(r1 = rk(yy, 1), r2 = rk(yy, 2)), 4)

**Exercise 9.5** (FREES exercise 8.2). (a) Derive the approximate relationship between $\text{DW}$ and $r_1$. (b) If $r_1=0.4$, what is $\text{DW}$ approximately?

**Solution.** (a) Proposition 9.18: $\text{DW}=2(1-r_1)-(a_1^2+a_T^2)/\sum_ta_t^2$ exactly, and the last term is two of the $T$ squares divided by their sum, small for large $T$ when no single $a_t^2$ carries a fixed share of the sum (for white noise with $\operatorname{\mathsf{E}} y_t^4<\infty$, for example, $T^{-1}\sum_ta_t^2\xrightarrow{\,p\,}\sigma^2$ by the proof of Proposition 9.19, while, with $z_t=y_t-\mu$ as there, $a_1^2\leqslant2(z_1^2+\bar z^2)$ with $z_1$ of fixed law, so $(a_1^2+a_T^2)/T\xrightarrow{\,p\,}0$); so $\text{DW}\approx2(1-r_1)$. (b) $\text{DW}\approx1.2$. ■

**Exercise 9.6** (FREES exercise 8.6, parts (a), (b)). For the $503$ daily returns of the S&P value-weighted index in 2005–2006: (a) comment on stationarity; (b) give summary statistics and, under a white noise model, forecasts for the first three trading days of 2007. Part (c), the significance of the autocorrelations, needs the sampling law of $r_k$, which is Unit 10's.

**Solution.**

In [ ]:
sd5 <- read.csv("data/frees/csv/SP500Daily.csv")
r56 <- sd5$vwretd[sd5$caldt >= 20050101 & sd5$caldt <= 20061231]
round(c(T = length(r56), mean = mean(r56), sd = sd(r56),
        min = min(r56), max = max(r56)), 5)
yr56 <- substr(sd5$caldt, 1, 4)[sd5$caldt >= 20050101 & sd5$caldt <= 20061231]
round(rbind(mean = tapply(r56, yr56, mean), sd = tapply(r56, yr56, sd)), 5)
round(mean(r56) + c(-1, 1) * qt(0.975, length(r56) - 1) * sd(r56) *
        sqrt(1 + 1 / length(r56)), 5)

(a) The returns fluctuate about a constant level with no visible trend or change in spread over the two years (means $0.00022$ in 2005 and $0.00060$ in 2006, both small against the daily standard deviations $0.00647$ and $0.00632$), consistent with stationarity. (b) Under white noise the forecast at every lead is $\bar y$ (Theorem 9.8), the same for the three days, with the printed $95\%$ interval for each under normal white noise. ■

**Exercise 9.7** (SOA sample question 3). For a random walk $y_t=y_0+c_1+\cdots+c_t$, $c_{11},\ldots,c_{19}$ are $2,3,5,3,4,2,4,1,2$, the average of $c_1,\ldots,c_{10}$ is $2$, and $\hat y_{19}$ is forecast from $y_{10}$. Calculate $y_{19}-\hat y_{19}$: (A) 1; (B) 2; (C) 3; (D) 8; (E) 18.

**Solution.** By Theorem 9.11(i) with $T=10$, $l=9$ and $\bar c=2$, $y_{19}-\hat y_{19}=\sum_{j=11}^{19}c_j-9\bar c=26-18=8$: (D). ■

**Exercise 9.8** (SOA sample question 4). For a random walk with $y_0=0$ and $y_1,\ldots,y_{10}=2,5,10,13,18,20,24,25,27,30$, calculate the standard error of the nine-step forecast $\hat y_{19}$: (A) $4/3$; (B) $4$; (C) $9$; (D) $12$; (E) $16$.

**Solution.** The differences are $2,3,5,3,5,2,4,1,2,3$, with mean $3$ and $s_c^2=16/9$, $s_c=4/3$. FREES's standard error $s_c\sqrt l=(4/3)\sqrt9=4$ is (B). It omits the uncertainty in $\bar c$: by Theorem 9.11(i) the estimated standard deviation of the error is $s_c\sqrt{l(1+l/m)}=(4/3)\sqrt{9\cdot1.9}=5.51$ with $m=10$, not among the options. ■

**Exercise 9.9** (SOA sample question 21). For a random walk $y_t=y_{t-1}+c_t$ with $\operatorname{\mathsf{E}} c_t=\mu_c$ and $\operatorname{var} c_t=\sigma_c^2$: which is true? I. If $\mu_c\neq0$, it is nonstationary in the mean. II. If $\sigma_c^2=0$, it is nonstationary in the variance. III. If $\sigma_c^2>0$, it is nonstationary in the variance. (A) none; (B) I and II only; (C) I and III only; (D) II and III only; (E) none of these.

**Solution.** Proposition 9.10: $\operatorname{\mathsf{E}} y_t=y_0+t\mu_c$ depends on $t$ iff $\mu_c\neq0$ (I true), and $\operatorname{var} y_t=t\sigma_c^2$ is constant ($0$) iff $\sigma_c^2=0$ (II false, III true): (C). ■

**Exercise 9.10** (SOA sample question 31). Which indicates that a nonstationary series can be represented as a random walk? I. A control chart shows a linear trend in time and increasing variability. II. The differenced series follows a white noise model. III. The standard deviation of the series exceeds that of the differenced series. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of these.

**Solution.** I: by Proposition 9.10 a random walk has mean linear in $t$ and variance $t\sigma_c^2$. II: the differences of a random walk are its white noise $c_t$ (Definition 9.9). III: by Proposition 9.13, $\operatorname{\mathsf{E}} s_y^2\geqslant\sigma^2(T+1)/6$, far above $\operatorname{\mathsf{E}} s_c^2=\sigma^2$ for moderate $T$. All three: (D). II is the defining property; I and III are consequences that a stationary series can share in part, so they indicate rather than prove. ■

**Exercise 9.11** (SOA sample question 38). Model L is $y_t=\beta_0+\beta_1t+\varepsilon_t$ and Model M is $y_t=y_0+\mu_ct+u_t$ with $u_t=\sum_{j\leqslant t}\varepsilon_j$, $\{\varepsilon_t\}$ white noise. Which is true? I. L is a linear trend model whose error is not a random walk. II. M is a random walk whose error is also a random walk. III. The comparison of L and M is not clear when $\mu_c=0$. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of these.

**Solution.** Proposition 9.14: L's error is white noise, M is a random walk with drift $\mu_c$ whose error $u_t$ is a random walk, and with $\mu_c=0$ M has a constant mean, so it competes only with L's special case $\beta_1=0$, which is why the comparison is not clear. All three: (D). ■

**Exercise 9.12** (SOA sample question 55). A random walk has $y_0,\ldots,y_7$ equal to $3$, $5$, $7$, $8$, $12$, $15$, $21$, $22$; it is fitted to the first five observations, with estimated mean of the white noise $2.25$, and evaluated on the last three by one-step forecast residuals. With $F$ the $\text{ME}$ and $G$ the $\text{MSE}$, calculate $|F-G|$: (A) 3.48; (B) 4.31; (C) 5.54; (D) 6.47; (E) 7.63.

**Solution.** By Definition 9.16 the one-step forecasts are $y_{t-1}+2.25$: $14.25$, $17.25$, $23.25$, with residuals $0.75$, $3.75$, $-1.25$. So $F=3.25/3=1.0833$ and $G=(0.5625+14.0625+1.5625)/3=5.3958$, and $|F-G|=4.3125$: (B). The $2.25$ is $(12-3)/4$, the mean of the four differences of $y_0,\ldots,y_4$. ■